In [17]:
# ===== Paths (edit if needed) =====
jobs_path = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/jobs_merged_it.xlsx"
cvs_path  = "/Users/nguyenduykhanh/Documents/GraduationProject/Recommendation_Project/recommendation/datasets/UpdatedResumeDataSet.csv"

# ===== Config =====
TOPN_RETRIEVE = 200     # lọc sơ ra 200 job giống CV nhất về mặt chữ trước - giống nhất
TOPK_FINAL    = 10      # trả về 10 job tốt nhất cho ứng viên xem

# Weights for final_score
W_SEM   = 0.60      # So chữ (TF-IDF)
W_SKILL = 0.25      # Nếu kỹ năng trùng nhiều
W_TAX   = 0.15      # Job đúng ngành (backend với backend)
W_SEN_P = 0.30      # dựa vào mức độ junior / mid / senior giữa CV và job (KINH NGHIỆM)


In [18]:
import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [16]:


jobs = pd.read_excel(jobs_path)
cvs  = pd.read_csv(cvs_path)

print("Jobs shape:", jobs.shape)
print("CVs shape:", cvs.shape)
print("Jobs columns:", list(jobs.columns))
print("CVs columns:", list(cvs.columns))

def clean_text(s: str) -> str:
    s = "" if s is None else str(s)
    s = s.lower()
    s = re.sub(r"\s+", " ", s).strip()
    return s

# ===== Adapter: make merged jobs compatible with notebook schema =====
# If using jobs_merged_it.xlsx: it has "description" instead of "cleaned_description"

# 1) Ensure we have a description source
if "cleaned_description" not in jobs.columns:
    if "description" in jobs.columns:
        jobs["cleaned_description"] = jobs["description"]
    elif "job_description" in jobs.columns:
        jobs["cleaned_description"] = jobs["job_description"]
    else:
        raise ValueError("jobs dataset missing a description column to create cleaned_description")

# 2) Ensure title exists
if "title" not in jobs.columns:
    # try common alternatives (optional)
    for alt in ["job_title", "Job Title", "Title"]:
        if alt in jobs.columns:
            jobs["title"] = jobs[alt]
            break
    else:
        raise ValueError("jobs dataset missing required column: title")

# 3) Fill NaN + cast
jobs["title"] = jobs["title"].fillna("").astype(str)
jobs["cleaned_description"] = jobs["cleaned_description"].fillna("").astype(str)

# optional columns
for col in ["company", "location", "source"]:
    if col in jobs.columns:
        jobs[col] = jobs[col].fillna("").astype(str)

# 4) Clean text
jobs["title"] = jobs["title"].apply(clean_text)
jobs["cleaned_description"] = jobs["cleaned_description"].apply(clean_text)

# 5) (Optional) Make a unified text field used later
jobs["job_text"] = jobs["title"] + " " + jobs["cleaned_description"]

# 6) Final check (NOW it will pass)
for col in ["title", "cleaned_description"]:
    if col not in jobs.columns:
        raise ValueError(f"jobs dataset missing required column: {col}")

display(jobs.head(3))
display(cvs.head(3))

job_texts = jobs["job_text"]

# CV text dùng cho similarity
# Tuỳ dataset CV của bạn, thường là cột Resume
if "Resume" in cvs.columns:
    cv_texts = cvs["Resume"]
elif "cleaned_resume" in cvs.columns:
    cv_texts = cvs["cleaned_resume"]
else:
    raise ValueError("CV dataset missing Resume text column")

resumes = cvs                # some functions use `resumes` name
job_texts = jobs["job_text"] # Step 2 expects this
cv_texts  = resumes["Resume"]  # Step 2 expects this (UpdatedResumeDataSet has Resume)

print("job_texts:", job_texts.shape, "cv_texts:", cv_texts.shape)

Jobs shape: (89277, 5)
CVs shape: (962, 2)
Jobs columns: ['title', 'description', 'company', 'location', 'source']
CVs columns: ['Category', 'Resume']


,title,description,company,location,source,cleaned_description,job_text
0,support technologist ii 2024-02216,**description and functions** ----------------...,State of Wyoming,"Rock Springs, WY",it,**description and functions** ----------------...,support technologist ii 2024-02216 **descripti...
1,computer technician,***if you would like to apply for this positio...,Medford Township Public Schools,"Medford, NJ",it,***if you would like to apply for this positio...,computer technician ***if you would like to ap...
2,it technician,i need a sped up recording converted to text. ...,Law Office of Ruben and Ruben LLC,,it,i need a sped up recording converted to text. ...,it technician i need a sped up recording conve...


,Category,Resume
0,Data Science,Skills * Programming Languages: Python (pandas...
1,Data Science,Education Details \r\nMay 2013 to May 2017 B.E...
2,Data Science,"Areas of Interest Deep Learning, Control Syste..."


job_texts: (89277,) cv_texts: (962,)


In [7]:
# ===== Step 2: Normalize text =====
# Chuyển hết về chữ thường
# Xoá ký tự rác, nhưng giữ lại: + ; # ; .
# Dọn khoảng trắng
# Xoá khoảng trắng thừa đầu/cuối

def norm_text(t: str) -> str:
    t = str(t).lower()
    t = re.sub(r"[^a-z0-9\s\+\#\.]", " ", t)   # keep + # . for c++, c#, node.js
    t = re.sub(r"\s+", " ", t)
    return t.strip()

job_texts_norm = job_texts.apply(norm_text)  # lấy từng job đã được chuẩn hoá chữ
cv_texts_norm  = cv_texts.apply(norm_text)   # lấy từng CV đã được chuẩn hoá chữ


In [8]:
# ===== Step 3: TF-IDF vectorization =====
tfidf = TfidfVectorizer(
    max_features=30000,     # đếm chữ và chỉ nhớ tối đa 30.000 từ/cụm từ quan trọng nhất
    ngram_range=(1,2),      # 1 chữ đơn và 2 chữ liền
    min_df=3,               # chữ nào xuất hiện dưới 3 job => bỏ
    stop_words="english"    # bỏ những từ vô nghĩa như: the, and, is, for,...
)

job_tfidf = tfidf.fit_transform(job_texts_norm) # đọc hết chữ all job, học xem chữ nào quan trọng
                                                # rồi biến mỗi job thành 1 dãy số
print("job_tfidf shape:", job_tfidf.shape)


job_tfidf shape: (89277, 30000)


In [9]:
# trả về top N job giống CV nhất (dựa trên chữ)
def retrieve_topn_jobs_for_cv_tfidf(cv_index: int, topn: int = TOPN_RETRIEVE):
    cv_vec = tfidf.transform([cv_texts_norm.iloc[cv_index]])   # CV đã được chuyển dãy số
    sims = cosine_similarity(cv_vec, job_tfidf)[0]  # độ tương đồng giữa job và CV
    idx = np.argsort(-sims)[:topn] # sắp xếp simil giảm dần

    cands = jobs.iloc[idx].copy() # Lấy thông tin job tương ứng với topN index
    cands["semantic_score"] = sims[idx] # Cột được thêm để biết job giống CV như nào
    return cands, idx


In [10]:
# ===== Step 5: Taxonomy / Seniority / Skills =====
# taxonomy nhãn chuẩn DANH MỤC
CATEGORY_TO_TAXON = {
  "Data Science": "data_ai",
  "Python Developer": "backend",
  "Java Developer": "backend",
  "DotNet Developer": "backend",
  "Web Designing": "frontend",
  "DevOps Engineer": "devops_cloud",
  "Testing": "qa",
  "Network Security Engineer": "security",
  "Database": "data_ai",
  "Hadoop": "data_ai",
  "ETL Developer": "data_ai",
}

# taxononmy nhãn chuẩn MỖI NGÀNH
TAXON_KEYWORDS = {
  "backend": ["backend","software engineer","application developer","api","rest","microservice","spring","django","flask","fastapi","node","express","golang","java",".net","c#"],
  "frontend": ["frontend","react","vue","angular","javascript","typescript","ui","css","html","next.js"],
  "data_ai": ["machine learning","ml","ai","nlp","deep learning","pytorch","tensorflow","etl","bi","warehouse","spark","hadoop","airflow"],
  "devops_cloud": ["devops","cloud","kubernetes","docker","aws","azure","gcp","terraform","ci/cd","jenkins","helm","ansible","sre","platform","prometheus","grafana"],
  "mobile": ["android","ios","swift","kotlin","flutter","react native"],
  "qa": ["qa","tester","testing","automation","selenium","cypress","playwright"],
  "security": ["security","soc","siem","pentest","vulnerability","owasp"],
}

# đoán ngành job: đếm keyw trong tỉtile và desc => tit được ưu tiên hơn
def classify_taxonomy_title_desc(title: str, desc: str) -> str:
    t_title = norm_text(title)
    t_desc  = norm_text(desc)

    scores = {tax: 0 for tax in TAXON_KEYWORDS} # tạo bảng điểm cho từng ngành
    for tax, kws in TAXON_KEYWORDS.items():
        scores[tax] = 3*sum(1 for kw in kws if kw in t_title) + 1*sum(1 for kw in kws if kw in t_desc)

    best = max(scores, key=scores.get)  # chọn ngành có điểm cao nhất
    return best if scores[best] > 0 else "other"    

# nếu có cột Cate thì dùng; còn không thì tự suy ra ngành từ nội dung CV
def cv_taxonomy(cv_index: int) -> str:
    if "Category" in resumes.columns:
        cat = resumes.iloc[cv_index]["Category"]
        mapped = CATEGORY_TO_TAXON.get(cat, None)
        if mapped:
            return mapped
    return classify_taxonomy_title_desc("", cv_texts_norm.iloc[cv_index])

# kinh nghiệm: nếu ngoài thì mid
def infer_seniority_from_title(title: str) -> str:
    t = title.lower()
    if any(x in t for x in ["intern", "internship", "trainee", "junior", "jr"]):
        return "junior"
    if any(x in t for x in ["senior", "sr", "lead", "principal", "staff", "manager", "director", "head"]):
        return "senior"
    return "mid"

# auto junior và mục đích là sẽ khong đợi ý job senior cho junior
def infer_seniority_from_cv(text: str) -> str:
    t = text.lower()
    if "intern" in t or "internship" in t:
        return "junior"
    if any(x in t for x in ["senior", "lead", "manager", "director", "principal", "staff"]):
        return "senior"
    return "junior"

# đo job cao hơn level CV bao nhiêu bậc và chỉ phạt khi job đòi level cao hơn CV ứng viên
def seniority_penalty(cv_sen: str, job_sen: str) -> float:
    order = {"junior":0, "mid":1, "senior":2}
    return max(0, order.get(job_sen,1) - order.get(cv_sen,0))

# danh sách ngôn ngữ được lấy từ taxonomy và bổ sung thêm
SKILL_VOCAB = set(sum(TAXON_KEYWORDS.values(), [])) | {
    "sql","mysql","postgres","mongodb","redis",
    "linux","git","jira",
    "docker","kubernetes","terraform",
    "python","java","javascript","typescript",
    "react","angular","vue",
}

# quét text; skill nào trong vocab xuất hiện thì lấy
def extract_skills(text: str) -> set:
    t = norm_text(text)
    return {kw for kw in SKILL_VOCAB if kw in t} # trả về tập skill
# dùng để tính matched_skills; matched_skills; skill_overlap

In [11]:
# ===== Step 6: Rerank + Explanation (Option 2) =====
def rerank_candidates(cv_index: int, cands_df: pd.DataFrame) -> pd.DataFrame:
    cv_text = cv_texts_norm.iloc[cv_index]      # lất text CV được chuẩn hoá
    cv_tax  = cv_taxonomy(cv_index)             # xác định CV thuộc ngành gì
    cv_sen  = infer_seniority_from_cv(cv_text)  # xác định CV thuộc level gì
    cv_sk   = extract_skills(cv_text)           # lấy ra tập skill của CV

    # copy 200 job; tit & dec không null và ghép chúng lại
    cands = cands_df.copy()
    cands["title"] = cands["title"].fillna("").astype(str)
    cands["cleaned_description"] = cands["cleaned_description"].fillna("").astype(str)
    job_text_series = cands["title"] + " " + cands["cleaned_description"]

    # gắn nhãn cho từng job gồm: tax (ngành); sen (level); skill (kỹ năng)
    cands["job_tax"] = cands.apply(lambda r: classify_taxonomy_title_desc(r["title"], r["cleaned_description"]), axis=1)
    cands["job_sen"] = cands["title"].apply(infer_seniority_from_title)
    cands["job_skills"] = job_text_series.apply(extract_skills)

    cands["cv_tax"] = cv_tax
    cands["cv_sen"] = cv_sen

    
    cands["tax_match"] = (cands["job_tax"] == cv_tax).astype(int)
    # ngành khớp =>  1, không khớp => 0
    def skill_overlap(job_sk: set) -> float:
        if not cv_sk:
            return 0.0
        return len(cv_sk & job_sk) / max(1, len(cv_sk))

    cands["skill_overlap"] = cands["job_skills"].apply(skill_overlap) # CV trùng skill bao nhiêu
    cands["sen_penalty"] = cands["job_sen"].apply(lambda js: seniority_penalty(cv_sen, js))
    # job cao level hơn CV → bị phạt

    cands["final_score"] = (
        W_SEM * cands["semantic_score"]     # giống chữ
        + W_SKILL * cands["skill_overlap"]  # trùng kỹ năng 
        + W_TAX * cands["tax_match"]        # đúng ngành
        - W_SEN_P * cands["sen_penalty"]    # lệch level thì trừ
    )

    def explain(job_sk: set):
        matched = sorted(list(cv_sk & job_sk))[:5] # những skill CV có và job cần
        missing = sorted(list(job_sk - cv_sk))[:5] # những skill job cần mà CV thiếu
        return matched, missing

    ex = cands["job_skills"].apply(explain)
    cands["matched_skills"] = ex.apply(lambda x: x[0])
    cands["missing_skills"] = ex.apply(lambda x: x[1])

    return cands.sort_values("final_score", ascending=False)


In [12]:
# ===== Step 7: Recommend (Improved) =====
def recommend_jobs_option2_tfidf(cv_index: int, top_k: int = TOPK_FINAL, topn_retrieve: int = TOPN_RETRIEVE) -> pd.DataFrame:
    cands, _ = retrieve_topn_jobs_for_cv_tfidf(cv_index, topn_retrieve) # lấy job lọc lần 1
    ranked = rerank_candidates(cv_index, cands)                         # lấy job lọc lần 2

    for col in ["company", "location"]:
        if col not in ranked.columns:
            ranked[col] = None

    cols = [
        "title","company","location",
        "semantic_score","final_score",
        "job_tax","job_sen","cv_tax","cv_sen",
        "matched_skills","missing_skills"
    ]
    return ranked[cols].head(top_k) # lấy topK job; final_score cao nhất


In [ ]:
demo_idx = 0
display(recommend_jobs_option2_tfidf(demo_idx, top_k=10))

# tiểu danh mục con

# làm slide, baseline chọn mô hình gì, cho thầy xem hình, baseline đấy có xu hướng cho năm nay hay không, tổng hợp thành slide giới thiệu tổng quát


# hệ gợi ý và taxonomy (chỉ tập trung vào hệ gợi ý)

# xử lý dữ liệu, xác định rõ ràng bài toán, xác định rõ baseline như thế nào, cố gắng tranining hết baseline, 

# quan trọng nhất là workflow, hình hệ thống và luồng của dữ liệu, có thể chi tiết là cách xử lý của dữ liệu;

# list mục lục bao gồm các chương

# nên thêm chương nào không hay chỉ gói gọn trong 4 chương


# giới thiệu web - giới thiệu dữ liệu - có 1 mục dữ liệu - dữ liệu trang nào, link nào

# xử lý dữ liệu như nào

# bài toán - mô hình đề xuất mô tả xem hệ thống tranning đi luồng như nào => workflow

# traiining cài đặt bao nhiêi vòng lặp  (tham số đặt trong mô hình)

# kết quả và thảo luận

# giới thiệu về sản phẩm, viết hướng dẫn, hướng phát triển, điểm đạt được,...

# kết luận

,title,company,location,semantic_score,final_score,job_tax,job_sen,cv_tax,cv_sen,matched_skills,missing_skills
71116,data scientist,Brigham Young University,"Provo, UT, US",0.178472,0.434167,data_ai,mid,data_ai,senior,"[ai, cloud, deep learning, docker, git]","[automation, aws, azure, bi, etl]"
69267,lead data scientist,AXIANS,US,0.154859,0.409582,data_ai,senior,data_ai,senior,"[ai, angular, cloud, css, deep learning]","[api, aws, azure, bi, django]"
71376,director of data science & artificial intellig...,Fannie Mae,"Washington, DC, US",0.161941,0.392998,data_ai,senior,data_ai,senior,"[ai, cloud, deep learning, docker, git]","[api, automation, aws, azure, bi]"
76844,data scientist ii- assurance analysis,Honeywell,"Kansas City, MO, US",0.179117,0.382470,data_ai,mid,data_ai,senior,"[ai, cloud, deep learning, git, machine learning]","[bi, hadoop, pytorch, spark, tensorflow]"
77021,data science - senior - consumer products & re...,EY,"New York, NY, US",0.161634,0.382397,data_ai,senior,data_ai,senior,"[ai, cloud, express, java, javascript]","[api, aws, bi, pytorch, software engineer]"
75185,principal data scientist,Surescripts,"Minneapolis, MN, USA",0.157012,0.379624,data_ai,senior,data_ai,senior,"[ai, cloud, deep learning, express, machine le...","[api, aws, bi, gcp, pytorch]"
74967,principal data scientist,Surescripts,US,0.155722,0.378850,data_ai,senior,data_ai,senior,"[ai, cloud, deep learning, express, machine le...","[api, aws, bi, gcp, pytorch]"
74091,data scientist,DrFirst,US,0.155008,0.378421,data_ai,mid,data_ai,senior,"[ai, cloud, deep learning, git, machine learning]","[api, aws, azure, bi, gcp]"
69445,junior data scientist,Iron EagleX,"Fayetteville, NC, US",0.167328,0.375397,data_ai,junior,data_ai,senior,"[ai, deep learning, docker, git, machine learn...","[api, backend, bi, devops, etl]"
74602,data scientist,LP Building Solutions,"Nashville, TN, USA",0.182664,0.374182,data_ai,mid,data_ai,senior,"[ai, deep learning, java, javascript, machine ...","[bi, ios, nlp, warehouse]"
